<a href="https://colab.research.google.com/github/wzy34/colab_projects/blob/main/names_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A MLP approach of generating names
Based on [A Neural Probabilistic Language Model](http://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

## Dataset preparation

In [ ]:
words = open("/content/drive/MyDrive/names.txt").read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [ ]:
len(words)

32033

In [ ]:
chars = sorted(list(set("".join(words))))
stoi = {char:i + 1 for i, char in enumerate(chars)}
stoi['.'] = 0
print(stoi)
itos = {i+1:s for i, s in enumerate(chars)}
itos[0] = '.'
print(itos)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [ ]:
block_size = 3
X, Y = [], []
for w in words[:5]:
  print(w)
  context = [0] * block_size
  for c in w + '.':
    ix = stoi[c]
    X.append(context)
    Y.append(ix)
    print("".join(itos[i] for i in context), "---->", itos[ix])
    # perform like a sliding window
    context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [ ]:
print(X)
print(Y)

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])
tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])


## Build look-up table C in the paper

In [ ]:
C = torch.randn((27, 2))
C

tensor([[-0.4719, -0.1203],
        [-0.7264, -0.4627],
        [ 0.3123, -0.5076],
        [ 0.6876,  0.4032],
        [-0.4197, -0.0743],
        [ 0.7646, -1.5905],
        [ 0.2830, -1.3581],
        [-0.3545,  0.4595],
        [ 1.0904, -0.3685],
        [ 0.4225, -1.6507],
        [-1.1448,  0.3122],
        [ 1.0375, -0.3441],
        [-1.0476, -0.4017],
        [ 0.4961, -0.2223],
        [-0.9634, -0.1807],
        [ 0.1506, -1.2430],
        [-0.4973,  1.3535],
        [ 0.2506, -0.7965],
        [ 0.5593, -0.1316],
        [-0.9311,  0.4323],
        [ 1.4072, -0.1229],
        [ 0.4252,  0.7450],
        [-0.3689, -2.7401],
        [ 0.3055, -1.2470],
        [-1.0989,  0.2753],
        [-0.1059, -0.5596],
        [ 0.1661, -0.4676]])

**Look up table can also be considered as the first layer of a neural network**

In [ ]:
C[5]

tensor([ 0.7646, -1.5905])

In [ ]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([ 0.7646, -1.5905])

In [ ]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [ ]:
print(emb)

tensor([[[-0.4719, -0.1203],
         [-0.4719, -0.1203],
         [-0.4719, -0.1203]],

        [[-0.4719, -0.1203],
         [-0.4719, -0.1203],
         [ 0.7646, -1.5905]],

        [[-0.4719, -0.1203],
         [ 0.7646, -1.5905],
         [ 0.4961, -0.2223]],

        [[ 0.7646, -1.5905],
         [ 0.4961, -0.2223],
         [ 0.4961, -0.2223]],

        [[ 0.4961, -0.2223],
         [ 0.4961, -0.2223],
         [-0.7264, -0.4627]],

        [[-0.4719, -0.1203],
         [-0.4719, -0.1203],
         [-0.4719, -0.1203]],

        [[-0.4719, -0.1203],
         [-0.4719, -0.1203],
         [ 0.1506, -1.2430]],

        [[-0.4719, -0.1203],
         [ 0.1506, -1.2430],
         [-1.0476, -0.4017]],

        [[ 0.1506, -1.2430],
         [-1.0476, -0.4017],
         [ 0.4225, -1.6507]],

        [[-1.0476, -0.4017],
         [ 0.4225, -1.6507],
         [-0.3689, -2.7401]],

        [[ 0.4225, -1.6507],
         [-0.3689, -2.7401],
         [ 0.4225, -1.6507]],

        [[-0.3689, -2

In [ ]:
torch.cat((emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]), 1).shape

torch.Size([32, 6])

In [ ]:
torch.cat(torch.unbind(emb, 1), 1).shape

torch.Size([32, 6])

In [ ]:
emb.view(32, 6)

tensor([[-0.4719, -0.1203, -0.4719, -0.1203, -0.4719, -0.1203],
        [-0.4719, -0.1203, -0.4719, -0.1203,  0.7646, -1.5905],
        [-0.4719, -0.1203,  0.7646, -1.5905,  0.4961, -0.2223],
        [ 0.7646, -1.5905,  0.4961, -0.2223,  0.4961, -0.2223],
        [ 0.4961, -0.2223,  0.4961, -0.2223, -0.7264, -0.4627],
        [-0.4719, -0.1203, -0.4719, -0.1203, -0.4719, -0.1203],
        [-0.4719, -0.1203, -0.4719, -0.1203,  0.1506, -1.2430],
        [-0.4719, -0.1203,  0.1506, -1.2430, -1.0476, -0.4017],
        [ 0.1506, -1.2430, -1.0476, -0.4017,  0.4225, -1.6507],
        [-1.0476, -0.4017,  0.4225, -1.6507, -0.3689, -2.7401],
        [ 0.4225, -1.6507, -0.3689, -2.7401,  0.4225, -1.6507],
        [-0.3689, -2.7401,  0.4225, -1.6507, -0.7264, -0.4627],
        [-0.4719, -0.1203, -0.4719, -0.1203, -0.4719, -0.1203],
        [-0.4719, -0.1203, -0.4719, -0.1203, -0.7264, -0.4627],
        [-0.4719, -0.1203, -0.7264, -0.4627, -0.3689, -2.7401],
        [-0.7264, -0.4627, -0.3689, -2.7

In [ ]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [ ]:
# -1: infer the size automatically
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
print(h)

tensor([[ 0.5761, -0.9135, -0.0551,  ..., -0.9962, -0.6353, -0.0077],
        [ 0.9253, -0.9996, -0.8598,  ..., -0.1738, -0.8067,  0.9977],
        [-0.3648, -0.9999, -0.9529,  ..., -0.9989, -0.9967,  0.8963],
        ...,
        [ 1.0000, -0.9511,  0.7867,  ...,  0.9731,  0.9939, -0.9996],
        [-0.3636, -0.9999, -0.9999,  ..., -0.0514,  0.0442,  0.9999],
        [-0.9992, -0.9893, -0.7703,  ..., -0.9991, -0.8339,  0.7282]])


In [ ]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [ ]:
logits = h @ W2 + b2

In [ ]:
logits.shape

torch.Size([32, 27])